# Applied Science Retail Analytics Project

This notebook presents an applied-science-focused analysis of the UCI Online Retail transactional dataset. The project studies revenue patterns, country-level sales distribution, order behavior, and customer segments using RFM scoring.

## Objectives

- Clean and validate retail transaction data
- Measure revenue, order value, and customer coverage
- Visualize monthly sales trends and high-performing countries/products
- Study the distribution of order values
- Build customer segments with Recency, Frequency, and Monetary analysis
- Show a reproducible analytics workflow that supports downstream modeling and business decision-making

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = ROOT / 'data' / 'raw' / 'online_retail.xlsx'
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.mplconfig'))
(ROOT / '.mplconfig').mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 140
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.titleweight'] = 'bold'

In [ ]:
df = pd.read_excel(DATA_PATH)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df['CustomerID'] = df['CustomerID'].astype('Int64')
df['Description'] = df['Description'].fillna('Unknown')

df = df.dropna(subset=['InvoiceDate', 'CustomerID'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

df['Revenue'] = df['Quantity'] * df['UnitPrice']
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Hour'] = df['InvoiceDate'].dt.hour

df.shape

In [ ]:
invoice_totals = df.groupby('InvoiceNo')['Revenue'].sum()
summary = {
    'records_after_cleaning': int(len(df)),
    'unique_customers': int(df['CustomerID'].nunique()),
    'countries_covered': int(df['Country'].nunique()),
    'study_period_start': df['InvoiceDate'].min().strftime('%Y-%m-%d'),
    'study_period_end': df['InvoiceDate'].max().strftime('%Y-%m-%d'),
    'total_revenue': round(float(df['Revenue'].sum()), 2),
    'average_order_value': round(float(invoice_totals.mean()), 2),
    'median_order_value': round(float(invoice_totals.median()), 2),
}
summary

In [ ]:
monthly = (
    df.groupby('InvoiceMonth')
    .agg(Revenue=('Revenue', 'sum'), Orders=('InvoiceNo', 'nunique'), Customers=('CustomerID', 'nunique'))
    .reset_index()
)
monthly['AverageOrderValue'] = monthly['Revenue'] / monthly['Orders']

plt.figure(figsize=(12, 6))
plt.plot(monthly['InvoiceMonth'], monthly['Revenue'], color='#1f77b4', linewidth=3, marker='o')
plt.fill_between(monthly['InvoiceMonth'], monthly['Revenue'], color='#a6d8ff', alpha=0.35)
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
country = (
    df.groupby('Country')
    .agg(Revenue=('Revenue', 'sum'), Orders=('InvoiceNo', 'nunique'), Customers=('CustomerID', 'nunique'))
    .sort_values('Revenue', ascending=False)
    .reset_index()
)
top_country = country[country['Country'] != 'United Kingdom'].head(10).sort_values('Revenue')

plt.figure(figsize=(12, 6))
plt.barh(top_country['Country'], top_country['Revenue'], color=sns.color_palette('viridis', len(top_country)))
plt.title('Top 10 Countries by Revenue (Excluding UK)')
plt.xlabel('Revenue')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

country.head(10)

In [ ]:
product = (
    df.groupby('Description')
    .agg(Revenue=('Revenue', 'sum'), Quantity=('Quantity', 'sum'), Orders=('InvoiceNo', 'nunique'))
    .sort_values('Revenue', ascending=False)
    .reset_index()
)
top_products = product.head(12).sort_values('Revenue')

plt.figure(figsize=(14, 8))
plt.barh(top_products['Description'], top_products['Revenue'], color=sns.color_palette('magma', len(top_products)))
plt.title('Top 12 Products by Revenue')
plt.xlabel('Revenue')
plt.ylabel('Product')
plt.tight_layout()
plt.show()

product.head(12)

In [ ]:
invoice_totals = df.groupby('InvoiceNo')['Revenue'].sum()

plt.figure(figsize=(12, 6))
sns.histplot(invoice_totals, bins=50, color='#ff7f0e', kde=True)
plt.title('Distribution of Order Values')
plt.xlabel('Order Value')
plt.ylabel('Number of Orders')
plt.xlim(0, np.quantile(invoice_totals, 0.99))
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = (
    df.assign(Weekday=df['InvoiceDate'].dt.day_name())
    .pivot_table(index='Weekday', columns='Hour', values='Revenue', aggfunc='sum')
    .reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday'])
    .fillna(0)
)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlGnBu')
plt.title('Revenue Heatmap by Weekday and Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Weekday')
plt.tight_layout()
plt.show()

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = (
    df.groupby('CustomerID')
    .agg(
        Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('Revenue', 'sum'),
    )
    .reset_index()
)

rfm['R_score'] = pd.qcut(rfm['Recency'].rank(method='first', ascending=False), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

conditions = [
    rfm['RFM_score'] >= 10,
    rfm['RFM_score'].between(7, 9),
    rfm['RFM_score'].between(5, 6),
    rfm['RFM_score'] <= 4,
]
choices = ['Champions', 'Loyal', 'Promising', 'At Risk']
rfm['Segment'] = np.select(conditions, choices, default='At Risk')

segment_counts = rfm['Segment'].value_counts().reindex(['Champions', 'Loyal', 'Promising', 'At Risk']).fillna(0)
segment_counts

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(segment_counts.index, segment_counts.values, color=['#00b894', '#0984e3', '#fdcb6e', '#d63031'])
plt.title('Customer Segments from RFM Analysis')
plt.xlabel('Segment')
plt.ylabel('Customers')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 8))
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Segment', alpha=0.75, s=70)
plt.title('Customer Value Map: Frequency vs Monetary')
plt.xlabel('Order Frequency')
plt.ylabel('Monetary Value')
plt.yscale('log')
plt.tight_layout()
plt.show()

## Conclusion

The dataset shows a highly concentrated retail environment with strong revenue concentration across a small set of countries, products, and customer segments. Monthly patterns, order-value variation, and the RFM segmentation together provide an applied-science-style view of customer behavior, feature-driven segmentation, and business performance.